# Klasy używane w algorytmach

In [75]:
class Point:
    
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __repr__(self):
        return f"Point({self.x}, {self.y})"

class Segment:
    
    def __init__(self, point1, point2):
        self.first_point = point1
        self.second_point = point2
        
    def reversed(self):
        return Segment(self.second_point, self.first_point)

# Rysowanie punktów

In [76]:
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.widgets import Button

In [77]:
class PolygonDrawer:
    def __init__(self):
        self.fig, self.ax = plt.subplots()
        self.ax.set_title("Kliknij, aby rysować punkty")
        self.points = []
        
        # Buttons
        self.clear_button_ax = self.fig.add_axes([0.81, 0, 0.1, 0.075])
        self.clear_button = Button(self.clear_button_ax, 'Wyczyść')
        self.clear_button.on_clicked(self.clear_polygon)
        
        # Click
        self.cid = self.fig.canvas.mpl_connect('button_press_event', self.onclick)
        self.draw_polygon()

    def onclick(self, event):
        if event.inaxes != self.ax:
            return
        self.points.append(Point(event.xdata, event.ydata))
        self.draw_polygon()

    def draw_polygon(self):
        xlim, ylim = self.ax.get_xlim(), self.ax.get_ylim()
        
        self.ax.clear()
        self.ax.set_title("Kliknij, aby rysować punkty")
        
        if self.points:
            self.ax.scatter([p.x for p in self.points], [p.y for p in self.points], color='blue')
        
        self.ax.set_xlim(xlim)
        self.ax.set_ylim(ylim)
        self.fig.canvas.draw()
    
    def clear_polygon(self, event):
        self.points = []
        self.ax.clear()
        self.ax.set_title("Kliknij, aby rysować punkty")
        self.fig.canvas.draw()


# Wizualizacja przebigu algorytmów

In [78]:
import os
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [79]:
class ConvexHullVisualizer:
    def __init__(self):
        
        self.frames = []

    def add_frame(self, points, hull_points, segments, current_point):
        frame_data = {
            "points": [(p.x, p.y) for p in points],
            "hull_points": [(p.x, p.y) for p in hull_points],
            "segments": [(segment.first_point.x, segment.first_point.y, segment.second_point.x, segment.second_point.y) for segment in segments],
            "current_point": (current_point.x, current_point.y)
        }
        self.frames.append(frame_data)

    def get_frames(self):
        return self.frames
    
    def create_animation(self):

        def update(frame):
            axs.clear()
            points = frame["points"]
            hull_points = frame["hull_points"]
            segments = frame["segments"]
            current_point = frame["current_point"]

            axs.scatter([p[0] for p in points], [p[1] for p in points], color='black')

            for seg in segments:
                axs.plot([seg[0], seg[2]], [seg[1], seg[3]], color='red', lw=2)

            if hull_points:
                if not segments:
                    axs.plot([p[0] for p in hull_points], [p[1] for p in hull_points], color='red', lw=2)
                axs.scatter([p[0] for p in hull_points], [p[1] for p in hull_points], color='red', s=100)

            axs.scatter(current_point[0], current_point[1], color='blue', s=100)
            axs.plot([hull_points[-1][0], current_point[0]], [hull_points[-1][1], current_point[1]], color='lightblue', lw=2)

            padding = 0.1
            x_coords = [p[0] for p in points]
            y_coords = [p[1] for p in points]
            x_min, x_max = min(x_coords), max(x_coords)
            y_min, y_max = min(y_coords), max(y_coords)

            axs.set_xlim(x_min - padding * (x_max - x_min), x_max + padding * (x_max - x_min))
            axs.set_ylim(y_min - padding * (y_max - y_min), y_max + padding * (y_max - y_min))

            axs.set_title("Wizualizacja otoczki wypukłej")

        fig, axs = plt.subplots()
        anim = FuncAnimation(fig, update, frames=self.frames, repeat=False, interval=500)
        plt.close(fig)

        return HTML(anim.to_jshtml())
